# r/soccer analytical pipeline on Google Colab (GPU)

**FIFA Communities chapter — `code/06-analysis-pipeline.py` run on a Colab GPU.** Orchestration only: every cell
calls the same command-line program that runs locally; nothing analytical lives in this notebook.
Design: `docs/session-04_2026-10-22/01_analytical-pipeline-design.md` (§10 for Colab); approvals and runbook:
`docs/session-05_2026-10-23/`.

**Runtime:** *Runtime → Change runtime type → GPU*, A100 40 GB preferred (L4 acceptable; T4 works in float16,
≈ 3× slower), high-RAM. In VS Code with the Colab extension: open this notebook, choose *New Colab Server → GPU*.

**Before the first run, once:** upload `data/yearly datasets/` (17 CSV files, 2.4 GB) to your Google Drive under
`MyDrive/fifa-reddit-communities/data/yearly datasets/`. Results (`results/analysis/`, `data/analysis/`), the
embedding cache and the model weights (`.hf-cache`) are kept on Drive, so a disconnect costs nothing finished.

**Order of operations (design §3):** bins → embed + topics per bin → **label** per bin → *co-author review*
(`topic_overrides.csv`) → **taxonomy** per bin → *review* (`category_overrides.csv`) → calibration of the outlier
threshold `T` on bin 1 and one large bin → `reassign` → sentiment → collect → copy back → S6 locally in R.

In [ ]:
# 1. Google Drive (the first cell tests whether drive.mount works from this kernel — the VS Code Colab
#    extension was not documented on this point; the fallback downloads a shared folder with gdown).
import os, subprocess, sys
DRIVE = '/content/drive/MyDrive/fifa-reddit-communities'    # adjust if you used another folder name
SHARED_DATA_URL = ''   # fallback only: a Drive *folder* link to 'data/yearly datasets' shared with the account
try:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.isdir(f'{DRIVE}/data/yearly datasets'), f'{DRIVE}/data/yearly datasets not found on Drive'
    print('Drive mounted; yearly files:', len(os.listdir(f'{DRIVE}/data/yearly datasets')))
except Exception as e:
    print('drive.mount failed or folder missing:', e)
    if not SHARED_DATA_URL:
        raise SystemExit('Set SHARED_DATA_URL to a shared Drive folder link and re-run this cell')
    os.makedirs(f'{DRIVE}/data', exist_ok=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run(['gdown', '--folder', SHARED_DATA_URL, '-O', f'{DRIVE}/data/yearly datasets'], check=True)

In [ ]:
# 2. Code from GitHub (public repository), data copied from Drive to the local SSD (Drive I/O is slow for large CSVs),
#    results / caches / weights symlinked to Drive so they persist.
import os, subprocess
REPO = '/content/fifa-reddit-communities'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/alannemi/fifa-reddit-communities', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
os.chdir(REPO)
os.makedirs('data', exist_ok=True)
if not os.path.isdir('data/yearly datasets'):
    subprocess.run(['cp', '-r', f'{DRIVE}/data/yearly datasets', 'data/yearly datasets'], check=True)
for local, remote in [('results/analysis', f'{DRIVE}/results/analysis'), ('data/analysis', f'{DRIVE}/data/analysis'),
                      ('results/topic-models/embeddings', f'{DRIVE}/results/topic-models/embeddings'), ('.hf-cache', f'{DRIVE}/.hf-cache')]:
    os.makedirs(remote, exist_ok=True); os.makedirs(os.path.dirname(local) or '.', exist_ok=True)
    if os.path.islink(local) or not os.path.exists(local):
        if os.path.islink(local): os.unlink(local)
        os.symlink(remote, local)
os.environ['HF_HOME'] = f'{REPO}/.hf-cache'; os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('working in', os.getcwd(), '| yearly files:', len(os.listdir('data/yearly datasets')))

In [ ]:
# 3. Environment: install ONLY the pinned packages on top of Colab's RAPIDS stack (review finding R8), then prove
#    that cuML, CUDA and the pins are intact. If `pip check` reports a broken requirement, fix that one package
#    explicitly (do not re-run a plain `pip install -r`).
import subprocess, sys, importlib
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-r', 'code/requirements-colab.txt'], check=True)
chk = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True); print(chk.stdout or 'pip check: OK')
import torch, cuml, bertopic, umap, hdbscan, sentence_transformers, transformers, numpy, sklearn, pandas
assert torch.cuda.is_available(), 'no CUDA device — change the runtime type to GPU'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| cuml', cuml.__version__)
print('bertopic', bertopic.__version__, '| umap', umap.__version__, '| hdbscan', hdbscan.__version__, '| sentence-transformers', sentence_transformers.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__, '| pandas', pandas.__version__)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])
# the program records the backend it actually used in each bin's config.json ("backend": "cuml" | "cpu")

In [ ]:
# helper: run one stage; `accel=True` prefixes `python -m cuml.accel` (GPU UMAP/HDBSCAN). Output streams below.
import subprocess, sys, shlex, time
def stage(args, accel=False, cuml_verbose=True):
    cmd = [sys.executable] + (['-m', 'cuml.accel'] + (['--verbose'] if cuml_verbose else []) if accel else []) + ['code/06-analysis-pipeline.py'] + shlex.split(args)
    print('>>>', ' '.join(cmd), flush=True); t0 = time.time()
    subprocess.run(cmd, check=True)
    print(f'<<< done in {(time.time() - t0) / 60:.1f} min', flush=True)

## S0 — period bins (every post, Σ must equal 1,302,021)

In [ ]:
stage('--stage bins')
import pandas as pd; print(pd.read_csv('data/analysis/period_bins_summary.csv').to_string(index=False))

## S1 + S2 — embeddings (mpnet, cached per year) and BERTopic per bin
Start with **bin 1** (the test case, design §12.1), read its `report.html` and `config.json` (`guard`, `backend`), then the
large-bin gate (bin 4 or 6, §12.2), then the rest. Each bin writes `results/analysis/bin-N/`.

In [ ]:
BINS = [1]                     # e.g. [1], then [4], then [2,3,5,6,7,8,9]
for b in BINS:
    stage(f'--stage embed --bin {b}')
    stage(f'--stage topics --bin {b}', accel=True)
import json
for b in BINS:
    cfg = json.load(open(f'results/analysis/bin-{b}/config.json'))['topics']
    print(f"bin {b}: {cfg['n_topics_raw']} topics, raw outliers {cfg['outlier_share_raw']:.1%}, backend {cfg['backend']['backend']}, guard {cfg['guard']['trials']} converged={cfg['guard']['converged']}")

## S3a — two-agent cluster labelling (resumable; re-run the cell after a disconnect)
Then **review pause**: co-authors read `results/analysis/bin-N/topics_labelled.csv` / `report.html` and fill
`topic_overrides.csv` (`final_name`, `merge_into`, `notes`). `needs_review` marks the topics the analysts did not agree on.

In [ ]:
for b in BINS:
    stage(f'--stage label --bin {b}')
import pandas as pd
for b in BINS:
    t = pd.read_csv(f'results/analysis/bin-{b}/topics_labelled.csv')
    print(f'bin {b}: {len(t)} topics, needs_review {t.needs_review.sum()}, modes {t.agreement_mode.value_counts().to_dict()}, rounds {t.round_iterations.value_counts().sort_index().to_dict()}')

## S3b — discourse taxonomy per bin (after the S3a review; any later S3a override requires `--force` here)
Then **review pause**: `categories.csv` → `category_overrides.csv`.

In [ ]:
for b in BINS:
    stage(f'--stage taxonomy --bin {b}')
import pandas as pd
for b in BINS:
    c = pd.read_csv(f'results/analysis/bin-{b}/categories.csv')
    print(f'bin {b}: {len(c)} categories [{c.agreement_mode.iloc[0]}, {c.round_iterations.iloc[0]} round(s)], repaired topics {c.n_repaired.sum()}')

## Outlier threshold `T` (decision D3; gate §12.3)
`results/analysis/bin-N/outlier_calibration/` holds the quantile table and one 100-post judgement sample per candidate
threshold (`belongs` column for the co-author). Once `T` is chosen, record it in `config.py` (`OUTLIER_THRESHOLD`) and in the
design's decisions log, then apply it here (seconds; no refit). Order: S2 → calibrate → reassign → S3a is preferred, but
reassignment never changes `topic_id_raw`, keywords or labels — only sizes, prevalence and the report.

In [ ]:
T = None                        # e.g. 0.55 after the co-author check
if T is not None:
    for b in BINS:
        stage(f'--stage reassign --bin {b} --outlier-threshold {T}')

## S4 — VADER sentiment (CPU, ≈ 15 min for the corpus; cached per year) and S5 — collect

In [ ]:
stage('--stage sentiment')
stage('--stage collect' + ('' if len(BINS) == 9 else ' --allow-partial'))
import json; print(json.dumps({k: v for k, v in json.load(open('data/analysis/post_variables.json')).items() if k != 'versions'}, indent=1))

## Copy back and finish locally
`results/analysis/` and `data/analysis/` are symlinked to Drive, so they are already there. Download them into the
repository (`results/analysis/` is tracked in git; `data/analysis/` goes to the shared data folder), then run stage S6:
```bash
Rscript code/07-tag-corpus.R && Rscript code/08-validate-analysis.R
```
and finalise codebook v3.0 (`docs/session-05_2026-10-23/03_runbook-session-06.md`).

In [ ]:
# optional: a quick inventory of what is on Drive now
import os
for root in ['results/analysis', 'data/analysis']:
    for dirpath, dirs, files in os.walk(root):
        if dirpath.count('/') <= 2: print(dirpath, len(files), 'files')